# ESMBiologist Demo

Zero-shot peptide scoring with ESM-2 embeddings.

In [1]:
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
import transformers

/home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define peptides

In [2]:
REFERENCE_PEPTIDE = "RLLKRFKHLFK"

candidate_peptides = [
    "RKRIHIGPGRAFYTT", # Random bad
    "KLLK", # Random bad
    "KWLNALLHHGLNCAKGVLA", # Random bad
    "ALLHHGLNCAKGVLA", # Random bad
    "WLNALLHHGLNCAKGVLA", # Random bad
    "FPWWWPF", # Random bad
    "VRRFPWWWPFLRR", # Random bad
    "RWRWRWFGGGRWRWRWF", # bad ?
    "LQFPVGRVHRLLRK", # Ok
    "TRSSRAGLQFPVGRVHR", # Ok
    "RLLKRFKHLFK", # B-36
    "KLLQTFKQIFR", # B-39
    "KLFKRWKHLFR", # B-35
]

print(f"Reference : {REFERENCE_PEPTIDE}")
print(f"Candidates: {len(candidate_peptides)} peptides")
for i in range(min(len(candidate_peptides)-1, 10)):
    print(f"  {candidate_peptides[i]}")

Reference : RLLKRFKHLFK
Candidates: 13 peptides
  RKRIHIGPGRAFYTT
  KLLK
  KWLNALLHHGLNCAKGVLA
  ALLHHGLNCAKGVLA
  WLNALLHHGLNCAKGVLA
  FPWWWPF
  VRRFPWWWPFLRR
  RWRWRWFGGGRWRWRWF
  LQFPVGRVHRLLRK
  TRSSRAGLQFPVGRVHR


## Instantiate ESMBiologist

In [3]:
biologist = ESMBiologistZscore(
    reference_peptide=REFERENCE_PEPTIDE,
    model_name="facebook/esm2_t33_650M_UR50D",
)
print(biologist)

Loading weights: 100%|██████████| 566/566 [00:00<00:00, 1241.01it/s, Materializing param=encoder.layer.32.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESMBiologistZscore(model='facebook/esm2_t33_650M_UR50D', layer=6, scoring='Z-Score L2')


## score_peptides

In [4]:
# Score all candidates
scores = biologist.score_peptides(candidate_peptides)

print(f"\n{'Peptide':<30} {'Score':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")


Peptide                          Score
----------------------------------------
RLLKRFKHLFK                    0.8910
KLFKRWKHLFR                    0.7642
KLLQTFKQIFR                    0.6848
LQFPVGRVHRLLRK                 0.5809
KLLK                           0.5086
RKRIHIGPGRAFYTT                0.4772
TRSSRAGLQFPVGRVHR              0.4613
KWLNALLHHGLNCAKGVLA            0.4555
WLNALLHHGLNCAKGVLA             0.4426
ALLHHGLNCAKGVLA                0.4366
VRRFPWWWPFLRR                  0.4186
FPWWWPF                        0.2141
RWRWRWFGGGRWRWRWF              0.1331


## predict_activity with an alternative reference

In [5]:
TEMP_REFERENCE = "KLFKRWKHLFR"

activity_scores = biologist.predict_activity(candidate_peptides, context=TEMP_REFERENCE)

print(f"{'Peptide':<30} {'Activity':>7}")
print("-" * 40)
for pep, sc in sorted(zip(candidate_peptides, activity_scores), key=lambda x: -x[1]):
    print(f"{pep:<30} {sc:.4f}")

Peptide                        Activity
----------------------------------------
KLFKRWKHLFR                    0.9002
RLLKRFKHLFK                    0.7698
KLLQTFKQIFR                    0.6355
LQFPVGRVHRLLRK                 0.5720
KLLK                           0.4927
RKRIHIGPGRAFYTT                0.4886
VRRFPWWWPFLRR                  0.4762
TRSSRAGLQFPVGRVHR              0.4663
KWLNALLHHGLNCAKGVLA            0.4445
WLNALLHHGLNCAKGVLA             0.4309
ALLHHGLNCAKGVLA                0.4200
FPWWWPF                        0.2198
RWRWRWFGGGRWRWRWF              0.1351
